In [ ]:
class SimpleTokenizerV1:

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {
            i: s for s, i in vocab.items()
        }

    def encode(self, text):
        preprocessed = re.split(
            r'([,.:;?_!"()\']|--|\s)',
            text
        )

        preprocessed = [
            item.strip()
            for item in preprocessed
            if item.strip()
        ]

        ids = [
            self.str_to_int[s]
            for s in preprocessed
        ]

        return ids

    def decode(self, ids):
        text = " ".join([
            self.int_to_str[i]
            for i in ids
        ])

        text = re.sub(
            r'\s+([,.?!"()\'])',
            r'\1',
            text
        )

        return text

In [ ]:
all_tokens = sorted(list(set(preprocessed)))

all_tokens.extend([
    "<|endoftext|>",
    "<|unk|>"
])

vocab = {
    token: integer
    for integer, token in enumerate(all_tokens)
}

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):

    def __init__(
        self,
        txt,
        tokenizer,
        max_length,
        stride
    ):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(
            txt,
            allowed_special={"<|endoftext|>"}
        )

        for i in range(
            0,
            len(token_ids) - max_length,
            stride
        ):
            input_chunk = token_ids[
                i:i + max_length
            ]

            target_chunk = token_ids[
                i + 1:i + max_length + 1
            ]

            self.input_ids.append(
                torch.tensor(input_chunk)
            )

            self.target_ids.append(
                torch.tensor(target_chunk)
            )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (
            self.input_ids[idx],
            self.target_ids[idx]
        )

In [ ]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(
        txt,
        tokenizer,
        max_length,
        stride
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [1]:
import torch
import tiktoken

from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):

    def __init__(
        self,
        txt,
        tokenizer,
        max_length,
        stride
    ):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(
            txt,
            allowed_special={"<|endoftext|>"}
        )

        for i in range(
            0,
            len(token_ids) - max_length,
            stride
        ):
            input_chunk = token_ids[
                i:i + max_length
            ]

            target_chunk = token_ids[
                i + 1:i + max_length + 1
            ]

            self.input_ids.append(
                torch.tensor(input_chunk)
            )

            self.target_ids.append(
                torch.tensor(target_chunk)
            )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return (
            self.input_ids[idx],
            self.target_ids[idx]
        )


def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
):
    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = GPTDatasetV1(
        txt,
        tokenizer,
        max_length,
        stride
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader


batch_size = 8
max_length = 4
output_dim = 256
vocab_size = 50257

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length,
    shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

token_embedding_layer = torch.nn.Embedding(
    vocab_size,
    output_dim
)

pos_embedding_layer = torch.nn.Embedding(
    max_length,
    output_dim
)

token_embeddings = token_embedding_layer(inputs)

pos_embeddings = pos_embedding_layer(
    torch.arange(max_length)
)

input_embeddings = (
    token_embeddings + pos_embeddings
)

NameError: name 'raw_text' is not defined